# 🚀 GPU-Accelerated Vector Database Engine
## Phase 2b — Shared Memory Tiled Distance Kernels

---

### What we verify here
| Step | Description |
|------|-------------|
| Cell 1 | GPU device info |
| Cell 2 | Clone repo and build |
| Cell 3 | Run all tests including new GPU tiled kernel tests |
| Cell 4 | Tiled vs basic kernel performance comparison |
| Cell 5 | Batch KNN benchmark |
| Cell 6 | Results summary |

> ⚠️ **Set runtime first:** `Runtime > Change runtime type > GPU > A100`

## Cell 1 — GPU Info

In [1]:
import subprocess

print('=== GPU Device ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
name, mem, cc = result.stdout.strip().split(',')
print(f'  GPU:               {name.strip()}')
print(f'  Memory:            {mem.strip()}')
print(f'  Compute Capability: sm_{cc.strip().replace(".", "")}')
print('\n✅ GPU ready')

=== GPU Device ===
Mon Apr 27 10:05:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

## Cell 2 — Clone and Build

In [6]:
import os, subprocess, time

REPO_URL = 'https://github.com/uday1o1/GPU-Accelerated-Vector-Database.git'
REPO_DIR = '/content/GPU-Accelerated-Vector-Database'
BUILD_DIR = f'{REPO_DIR}/build'

# Clone or pull
if os.path.exists(REPO_DIR):
    print('Pulling latest...')
    os.chdir(REPO_DIR)
    os.system('git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

print(f'Commit: ', end='')
os.system('git log --oneline -1')

# Install deps
print('\nInstalling dependencies...')
subprocess.run(
    'apt-get install -y cmake ninja-build libopenmpi-dev 2>&1 | tail -3',
    shell=True, capture_output=True
)
print('done')

# Configure
os.makedirs(BUILD_DIR, exist_ok=True)
os.chdir(BUILD_DIR)

print('\n=== CMake Configure ===')
result = subprocess.run(
    ['cmake', '..', '-DCMAKE_BUILD_TYPE=Release', '-G', 'Unix Makefiles'],
    capture_output=True, text=True
)
for line in result.stdout.splitlines():
    if any(k in line for k in ['CUDA', 'Version', 'Build type', '===']):
        print(f'  {line.strip()}')
if result.returncode != 0:
    print('FAILED:'); print(result.stderr[-2000:])
    raise RuntimeError('Configure failed')
print('✅ Configure OK')

# Build
print('\n=== Build ===')
t0 = time.time()
result = subprocess.run(
    ['cmake', '--build', '.', '-j8'],
    capture_output=True, text=True
)
elapsed = time.time() - t0
for line in result.stdout.strip().splitlines()[-10:]:
    print(f'  {line}')
if result.returncode != 0:
    print('FAILED:'); print(result.stderr[-3000:])
    raise RuntimeError('Build failed')
print(f'\n✅ Build OK in {elapsed:.1f}s')

# Check binaries
print('\n=== Build artifacts ===')
for path in [
    f'{BUILD_DIR}/tests/test_vectorstore',
    f'{BUILD_DIR}/tests/test_hnsw',
    f'{BUILD_DIR}/tests/test_gpu_kernels',
    f'{BUILD_DIR}/tests/test_batch_distance',
    f'{BUILD_DIR}/src/libvectordb_core.a',
    f'{BUILD_DIR}/src/libvectordb_cuda.a',
]:
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {os.path.basename(path)}')

Pulling latest...
Commit: 
Installing dependencies...
done

=== CMake Configure ===
  -- CUDA found: /usr/local/cuda/bin/nvcc
  -- === VectorDB Engine ===
  --   Version   : 0.1.0
  --   CUDA      : TRUE
  --   Build type: Release
✅ Configure OK

=== Build ===
  [ 29%] Built target gtest_main
  [ 37%] Built target vectordb_py
  [ 58%] Built target vectordb_cuda
  [ 66%] Built target test_vectorstore
  [ 75%] Built target test_hnsw
  [ 87%] Built target test_gpu_kernels
  [ 91%] Building CXX object tests/CMakeFiles/test_batch_distance.dir/test_batch_distance.cpp.o
  [ 95%] Linking CUDA device code CMakeFiles/test_batch_distance.dir/cmake_device_link.o
  [100%] Linking CXX executable test_batch_distance
  [100%] Built target test_batch_distance

✅ Build OK in 2.5s

=== Build artifacts ===
  ✅  test_vectorstore
  ✅  test_hnsw
  ✅  test_gpu_kernels
  ✅  test_batch_distance
  ✅  libvectordb_core.a
  ✅  libvectordb_cuda.a


## Cell 3 — Run All Tests
Runs all 4 test binaries. The new `test_batch_distance` now includes GPU tiled kernel tests.

In [7]:
import subprocess, os

BUILD_DIR = '/content/GPU-Accelerated-Vector-Database/build'

test_binaries = [
    (f'{BUILD_DIR}/tests/test_vectorstore',    'VectorStore Unit Tests'),
    (f'{BUILD_DIR}/tests/test_hnsw',           'HNSW Index Tests'),
    (f'{BUILD_DIR}/tests/test_gpu_kernels',    'GPU Kernel Agreement Tests'),
    (f'{BUILD_DIR}/tests/test_batch_distance', 'Tiled Distance Kernel Tests'),
]

all_passed = True
summary = []

for binary, label in test_binaries:
    print(f'\n=== {label} ===')
    if not os.path.exists(binary):
        print(f'❌ Missing: {binary}')
        all_passed = False
        continue
    result = subprocess.run([binary], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f'❌ FAILED:\n{result.stderr}')
        all_passed = False
        summary.append(f'❌ {label}')
    else:
        for line in result.stdout.splitlines():
            if 'PASSED' in line:
                summary.append(f'✅ {label}: {line.strip()}')

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
for s in summary:
    print(f'  {s}')
print()
print('✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED')


=== VectorStore Unit Tests ===
Running main() from /content/GPU-Accelerated-Vector-Database/build/_deps/googletest-src/googletest/src/gtest_main.cc
[==========] Running 12 tests from 1 test suite.
[----------] Global test environment set-up.
[----------] 12 tests from VectorStoreTest
[ RUN      ] VectorStoreTest.ConstructsCorrectly
[       OK ] VectorStoreTest.ConstructsCorrectly (0 ms)
[ RUN      ] VectorStoreTest.ThrowsOnZeroDim
[       OK ] VectorStoreTest.ThrowsOnZeroDim (0 ms)
[ RUN      ] VectorStoreTest.InsertSingleVector
[       OK ] VectorStoreTest.InsertSingleVector (0 ms)
[ RUN      ] VectorStoreTest.InsertWrongDimThrows
[       OK ] VectorStoreTest.InsertWrongDimThrows (0 ms)
[ RUN      ] VectorStoreTest.InsertBatchAssignsSequentialIds
[       OK ] VectorStoreTest.InsertBatchAssignsSequentialIds (0 ms)
[ RUN      ] VectorStoreTest.L2SearchReturnsClosest
[       OK ] VectorStoreTest.L2SearchReturnsClosest (0 ms)
[ RUN      ] VectorStoreTest.L2SearchRanksByDistance
[       O

## Cell 4 — Tiled vs Basic Kernel Performance
Compares throughput of the Phase 1b basic kernel vs Phase 2b shared memory tiled kernel.
The tiled kernel should be faster due to reduced global memory bandwidth.

In [9]:
import os, subprocess

REPO  = '/content/GPU-Accelerated-Vector-Database'
BUILD = f'{REPO}/build'

os.makedirs(f'{REPO}/benchmarks', exist_ok=True)

perf_src = '''
#include <iostream>
#include <chrono>
#include <vector>
#include <random>
#include <iomanip>
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include "cuda/kernels/distance_kernels.cuh"
#include "cuda/kernels/batch_distance.cuh"

using Clock = std::chrono::high_resolution_clock;

std::vector<float> rand_vecs(int N, int dim, int seed=42) {
    std::mt19937 rng(seed);
    std::uniform_real_distribution<float> d(-1.0f, 1.0f);
    std::vector<float> v(N*dim);
    for (auto& x : v) x = d(rng);
    return v;
}

void convert_fp16(const float* src, __half* dst, int n) {
    float* d_tmp;
    cudaMalloc(&d_tmp, n*sizeof(float));
    cudaMemcpy(d_tmp, src, n*sizeof(float), cudaMemcpyHostToDevice);
    vectordb::cuda::launch_fp32_to_fp16(d_tmp, dst, n);
    cudaDeviceSynchronize();
    cudaFree(d_tmp);
}

double benchmark_kernel(bool tiled, __half* d_q, __half* d_vecs,
                        float* d_dists, int N, int dim, int REPS) {
    // Warmup
    for (int i=0;i<5;i++) {
        if (tiled) vectordb::cuda::launch_tiled_l2_distance(d_q, d_vecs, d_dists, N, dim);
        else       vectordb::cuda::launch_l2_distance(d_q, d_vecs, d_dists, N, dim);
    }
    cudaDeviceSynchronize();

    auto t0 = Clock::now();
    for (int i=0;i<REPS;i++) {
        if (tiled) vectordb::cuda::launch_tiled_l2_distance(d_q, d_vecs, d_dists, N, dim);
        else       vectordb::cuda::launch_l2_distance(d_q, d_vecs, d_dists, N, dim);
    }
    cudaDeviceSynchronize();
    auto t1 = Clock::now();
    return std::chrono::duration<double, std::milli>(t1-t0).count() / REPS;
}

int main() {
    const int REPS = 100;

    std::cout << std::string(60, \'=\') << std::endl;
    std::cout << "  Phase 2b Tiled vs Basic Kernel Benchmark" << std::endl;
    std::cout << std::string(60, \'=\') << std::endl;

    for (int dim : {128, 256, 512}) {
        for (int N : {10000, 100000}) {
            auto h_q    = rand_vecs(1, dim, 1);
            auto h_vecs = rand_vecs(N, dim, 2);

            __half *d_q, *d_vecs;
            float  *d_dists;
            cudaMalloc(&d_q,     dim*sizeof(__half));
            cudaMalloc(&d_vecs,  N*dim*sizeof(__half));
            cudaMalloc(&d_dists, N*sizeof(float));

            convert_fp16(h_q.data(),    d_q,    dim);
            convert_fp16(h_vecs.data(), d_vecs, N*dim);

            double basic_ms = benchmark_kernel(false, d_q, d_vecs, d_dists, N, dim, REPS);
            double tiled_ms = benchmark_kernel(true,  d_q, d_vecs, d_dists, N, dim, REPS);

            double speedup = basic_ms / tiled_ms;
            std::cout << std::fixed << std::setprecision(3);
            std::cout << "  N=" << std::setw(7) << N
                      << "  dim=" << std::setw(4) << dim
                      << "  basic=" << std::setw(7) << basic_ms << "ms"
                      << "  tiled=" << std::setw(7) << tiled_ms << "ms"
                      << "  speedup=" << std::setprecision(2) << speedup << "x"
                      << std::endl;

            cudaFree(d_q); cudaFree(d_vecs); cudaFree(d_dists);
        }
    }
    std::cout << std::string(60, \'=\') << std::endl;
    return 0;
}
'''

src_path = f'{REPO}/benchmarks/phase2b_tiled_perf.cpp'
out_path = f'{BUILD}/phase2b_tiled_perf'

with open(src_path, 'w') as f:
    f.write(perf_src)

cuda_lib = subprocess.run(
    'dirname $(find /usr/local/cuda -name "libcudart.so" 2>/dev/null | head -1)',
    shell=True, capture_output=True, text=True
).stdout.strip()

compile_cmd = (
    f'nvcc -O3 -std=c++17 -DCUDA_ENABLED '
    f'-I{REPO}/include '
    f'-I/usr/local/cuda/include '
    f'{src_path} '
    f'{BUILD}/src/libvectordb_core.a '
    f'{BUILD}/src/libvectordb_cuda.a '
    f'-o {out_path} '
    f'-L{cuda_lib} -lcudart'
)

print('Compiling benchmark...')
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(f'❌ Compile failed:\n{result.stderr[-2000:]}')
else:
    print('✅ Compiled\n')
    result = subprocess.run([out_path], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(f'STDERR: {result.stderr}')

Compiling benchmark...
✅ Compiled

  Phase 2b Tiled vs Basic Kernel Benchmark
  N=  10000  dim= 128  basic=  0.030ms  tiled=  0.031ms  speedup=0.97x
  N= 100000  dim= 128  basic=  0.112ms  tiled=  0.112ms  speedup=1.00x
  N=  10000  dim= 256  basic=  0.058ms  tiled=  0.060ms  speedup=0.97x
  N= 100000  dim= 256  basic=  0.221ms  tiled=  0.220ms  speedup=1.01x
  N=  10000  dim= 512  basic=  0.113ms  tiled=  0.116ms  speedup=0.97x
  N= 100000  dim= 512  basic=  0.439ms  tiled=  0.420ms  speedup=1.04x



## Cell 5 — Batch KNN Benchmark
Tests the batch KNN kernel: compute K nearest neighbors for multiple queries simultaneously.

In [10]:
import os, subprocess

REPO  = '/content/GPU-Accelerated-Vector-Database'
BUILD = f'{REPO}/build'

knn_src = '''
#include <iostream>
#include <chrono>
#include <vector>
#include <random>
#include <iomanip>
#include <algorithm>
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include "cuda/kernels/distance_kernels.cuh"
#include "cuda/kernels/batch_distance.cuh"
#include "core/VectorStore.h"

using namespace vectordb;
using Clock = std::chrono::high_resolution_clock;

std::vector<float> rand_vecs(int N, int dim, int seed=42) {
    std::mt19937 rng(seed);
    std::uniform_real_distribution<float> d(-1.0f, 1.0f);
    std::vector<float> v(N*dim);
    for (auto& x : v) x = d(rng);
    return v;
}

void to_fp16_device(const float* src, __half* dst, int n) {
    float* tmp;
    cudaMalloc(&tmp, n*sizeof(float));
    cudaMemcpy(tmp, src, n*sizeof(float), cudaMemcpyHostToDevice);
    cuda::launch_fp32_to_fp16(tmp, dst, n);
    cudaDeviceSynchronize();
    cudaFree(tmp);
}

int main() {
    const int DIM = 128;
    const int N   = 10000;
    const int Q   = 100;
    const int K   = 10;

    std::cout << std::string(60, \'=\') << std::endl;
    std::cout << "  Phase 2b Batch KNN Benchmark" << std::endl;
    std::cout << "  N=" << N << "  Q=" << Q << "  K=" << K << "  dim=" << DIM << std::endl;
    std::cout << std::string(60, \'=\') << std::endl << std::endl;

    auto h_vecs    = rand_vecs(N, DIM, 1);
    auto h_queries = rand_vecs(Q, DIM, 2);

    __half *d_vecs, *d_queries;
    uint32_t *d_ids;
    float *d_dists;

    cudaMalloc(&d_vecs,    N*DIM*sizeof(__half));
    cudaMalloc(&d_queries, Q*DIM*sizeof(__half));
    cudaMalloc(&d_ids,     Q*K*sizeof(uint32_t));
    cudaMalloc(&d_dists,   Q*K*sizeof(float));

    to_fp16_device(h_vecs.data(),    d_vecs,    N*DIM);
    to_fp16_device(h_queries.data(), d_queries, Q*DIM);

    // Warmup
    for (int i=0;i<3;i++)
        cuda::launch_batch_knn(d_queries, d_vecs, d_ids, d_dists, Q, N, K, DIM);
    cudaDeviceSynchronize();

    // Time batch KNN
    const int REPS = 50;
    auto t0 = Clock::now();
    for (int i=0;i<REPS;i++)
        cuda::launch_batch_knn(d_queries, d_vecs, d_ids, d_dists, Q, N, K, DIM);
    cudaDeviceSynchronize();
    auto t1 = Clock::now();
    double ms = std::chrono::duration<double,std::milli>(t1-t0).count()/REPS;
    double qps = Q / (ms/1000.0);

    // CPU baseline: sequential single-query search
    VectorStoreConfig cfg(DIM, MetricType::L2, N+10);
    VectorStore store(cfg);
    for (int i=0;i<N;i++) {
        Vector v(h_vecs.data()+i*DIM, h_vecs.data()+(i+1)*DIM);
        store.insert(v);
    }
    auto tc0 = Clock::now();
    for (int q=0;q<Q;q++) {
        Vector qv(h_queries.data()+q*DIM, h_queries.data()+(q+1)*DIM);
        store.search(qv, K);
    }
    auto tc1 = Clock::now();
    double cpu_ms = std::chrono::duration<double,std::milli>(tc1-tc0).count();
    double cpu_qps = Q / (cpu_ms/1000.0);

    std::cout << std::fixed << std::setprecision(1);
    std::cout << "  CPU sequential:  " << (int)cpu_qps << " QPS  (" << cpu_ms << " ms total)" << std::endl;
    std::cout << "  GPU batch KNN:   " << (int)qps     << " QPS  (" << ms     << " ms/batch)" << std::endl;
    std::cout << "  Speedup:         " << std::setprecision(1) << (qps/cpu_qps) << "x" << std::endl;
    std::cout << std::string(60, \'=\') << std::endl;

    cudaFree(d_vecs); cudaFree(d_queries);
    cudaFree(d_ids);  cudaFree(d_dists);
    return 0;
}
'''

src_path = f'{REPO}/benchmarks/phase2b_batch_knn.cpp'
out_path = f'{BUILD}/phase2b_batch_knn'

with open(src_path, 'w') as f:
    f.write(knn_src)

cuda_lib = subprocess.run(
    'dirname $(find /usr/local/cuda -name "libcudart.so" 2>/dev/null | head -1)',
    shell=True, capture_output=True, text=True
).stdout.strip()

compile_cmd = (
    f'nvcc -O3 -std=c++17 -DCUDA_ENABLED '
    f'-I{REPO}/include '
    f'-I/usr/local/cuda/include '
    f'{src_path} '
    f'{BUILD}/src/libvectordb_core.a '
    f'{BUILD}/src/libvectordb_cuda.a '
    f'-o {out_path} '
    f'-L{cuda_lib} -lcudart'
)

print('Compiling batch KNN benchmark...')
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(f'❌ Compile failed:\n{result.stderr[-2000:]}')
else:
    print('✅ Compiled\n')
    result = subprocess.run([out_path], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(f'STDERR: {result.stderr}')

Compiling batch KNN benchmark...
✅ Compiled

  Phase 2b Batch KNN Benchmark
  N=10000  Q=100  K=10  dim=128

  CPU sequential:  597 QPS  (167.4 ms total)
  GPU batch KNN:   88951 QPS  (1.1 ms/batch)
  Speedup:         148.9x



## Cell 6 — Phase 2b Results

Fill in after running all cells:

| Item | Result |
|------|--------|
| GPU device | |
| test_vectorstore | /12 passing |
| test_hnsw | /12 passing |
| test_gpu_kernels | /7 passing |
| test_batch_distance (CPU) | /3 passing |
| test_batch_distance (GPU tiled L2) | pass / fail |
| test_batch_distance (GPU tiled cosine) | pass / fail |
| test_batch_distance (ranking preserved) | pass / fail |
| test_batch_distance (tiled vs basic agree) | pass / fail |
| Tiled speedup vs basic (dim=128, N=100k) | x |
| Batch KNN GPU QPS | |
| Batch KNN speedup vs CPU | x |

---
### Next: Phase 2c — CUTLASS Matrix Multiplication
Once all cells pass, go back to Claude with your results table.